##Fixed size Chunking

In [2]:
from langchain_text_splitters import CharacterTextSplitter


In [ ]:
dummy_text = "Success isn't just about what you achieve**, it's about how you inspire others to dream, and grow along"

In [ ]:
splitter = CharacterTextSplitter(
    chunk_size=15,
    chunk_overlap=5,
    separator="*"
)

In [23]:
fixed_chunks = splitter.split_text(dummy_text)

In [24]:
print(fixed_chunks)

["Success isn't j", "n't just about", 'bout what you a', 'you achieve**,', "e**, it's about", 'about how you i', 'you inspire oth', 'e others to dre', 'o dream, and gr', 'nd grow along']


##RecursiveChunking

In [47]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [48]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=20,
    chunk_overlap=5,
    separators=[r"\*\*", r"(?<=[.!?,])\s+", r"\s+", r""],
    is_separator_regex=True
)

In [49]:
recursive_chunks = splitter.split_text(dummy_text)

In [50]:
print(recursive_chunks)

["Success isn't just", 'just about.', 'what you achieve', '**,', "it's about how you", 'you inspire others', 'to dream,', 'and grow along']


In [ ]:
##Hybrid Chunking = RecursiveChunking + SemanticChunking

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
#Step 1 : RecursiveChunking

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 10,
    chunk_overlap = 2,
    separators = []
)

#A:  My Name is Shubham Kumar
#B:  I like to play games
#C:  Stock Market is down today
#D:  Apple stocks are negative
#E:  Siddhart went to office

# A->B : 0.67 (Distance between A to B) #Distance calculate using the cosine similaity search
# B->C : 0.87 (Distance between B to C)
# C->D : 0.57 (Distance between C to D)
# D->E : 0.90 (Distance between D to E)

In [70]:
recursive_chunks = recursive_splitter.split_text(dummy_text)
print("Number of chunks using recursiveChunking :",len(recursive_chunks))

Number of chunks using recursiveChunking : 14


In [71]:
#now we have to implement the semanticChunking
from langchain_openai import OpenAIEmbeddings

embeddings =OpenAIEmbeddings()

In [72]:
from langchain_experimental.text_splitter import SemanticChunker

In [ ]:
semantic_chunker = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=70
)
# [0.45,0.60,0.75,0.80] 
'''  
70/100*4 = 2.8 
round(2.8) = 3
index(3) = 0.80
'''

In [ ]:
#now the chunks we have got using the recursiveChunking 
#we have to chunk it through the semantic chunking

final_chunks = []

for chunk in recursive_chunks:
    semantic_sub_chunks = semantic_chunker.split_text(chunk)
    final_chunks.extend(semantic_sub_chunks)





In [68]:
#print the final refined Recursive + SemanticChunking

for i,chunk in enumerate(final_chunks):
    print(f"Chunk: {i+1}")
    print(chunk[:100])

Chunk: 1
Success isn't just about what you achieve, it's
Chunk: 2
you achieve, it's about how you inspire others to
Chunk: 3
inspire others to dream, and grow along
